In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

In [0]:
# initial csv dataset uploaded 
master_df = spark.table("customer_master")

master_df.show(10)

+-----------+------+-------------+---+----------+
|customer_id|  name|         city|age|membership|
+-----------+------+-------------+---+----------+
|        101|  Neha|  Bhubaneswar| 32|    Silver|
|        102| Nisha|       Jaipur| 23|      Gold|
|        103|  Aman|       Mysuru| 32|      Gold|
|        104| Sneha|       Ranchi| 27|    Bronze|
|        105| Rahul|       Mumbai| 24|    Bronze|
|        106| Akash|Visakhapatnam| 21|      Gold|
|        107|Sanjay|       Kanpur| 31|      Gold|
|        108| Mohit|        Surat| 24|    Silver|
|        109| Tanvi|      Lucknow| 33|    Bronze|
|        110|  NULL|       Kanpur| 27|    Silver|
+-----------+------+-------------+---+----------+
only showing top 10 rows


In [0]:
# saving it as delta table
master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("customers")

In [0]:
# read the table
delta_df = spark.table("customers")

delta_df.show(10)

+-----------+------+-------------+---+----------+
|customer_id|  name|         city|age|membership|
+-----------+------+-------------+---+----------+
|        101|  Neha|  Bhubaneswar| 32|    Silver|
|        102| Nisha|       Jaipur| 23|      Gold|
|        103|  Aman|       Mysuru| 32|      Gold|
|        104| Sneha|       Ranchi| 27|    Bronze|
|        105| Rahul|       Mumbai| 24|    Bronze|
|        106| Akash|Visakhapatnam| 21|      Gold|
|        107|Sanjay|       Kanpur| 31|      Gold|
|        108| Mohit|        Surat| 24|    Silver|
|        109| Tanvi|      Lucknow| 33|    Bronze|
|        110|  NULL|       Kanpur| 27|    Silver|
+-----------+------+-------------+---+----------+
only showing top 10 rows


In [0]:
# removing null and duplicate values and showing the cleaned table
clean_df = (
    spark.table("customers")
    .dropna()
    .dropDuplicates()
)

clean_df.show(10)

+-----------+------+--------+---+----------+
|customer_id|  name|    city|age|membership|
+-----------+------+--------+---+----------+
|        103|  Aman|  Mysuru| 32|      Gold|
|        107|Sanjay|  Kanpur| 31|      Gold|
|        154| Sonia|  Mumbai| 33|      Gold|
|        186|  Riya|Guwahati| 26|    Bronze|
|        122| Divya|  Mysuru| 25|      Gold|
|        124| Mohit|Guwahati| 24|    Bronze|
|        132| Arjun|    Pune| 24|      Gold|
|        134| Tanvi|   Surat| 35|      Gold|
|        137|  Neha|    Pune| 31|    Bronze|
|        142|Simran|  Nashik| 31|    Silver|
+-----------+------+--------+---+----------+
only showing top 10 rows


In [0]:
# overwrite the delta table
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customers")

In [0]:
# creation of incremental dataset
new_df = spark.table("customer_incremental")

new_df.show(10)

+-----------+----------------+-------------+---+----------+
|customer_id|            name|         city|age|membership|
+-----------+----------------+-------------+---+----------+
|        103|    Aman Updated|       Mysuru| 33|  Platinum|
|        110|Updated Customer|       Kanpur| 28|  Platinum|
|        125|   Manoj Updated|       Nagpur| 26|  Platinum|
|        140|   Arjun Updated|   Chandigarh| 35|  Platinum|
|        150|Ashutosh Updated|Visakhapatnam| 27|  Platinum|
|        175|   Rohit Updated|    Hyderabad| 26|  Platinum|
|        180|   Divya Updated|       Nagpur| 36|  Platinum|
|        190|   Ankit Updated|       Mysuru| 35|  Platinum|
|        195|   Arjun Updated|      Chennai| 32|  Platinum|
|        200|    Aman Updated|       Mumbai| 30|  Platinum|
+-----------+----------------+-------------+---+----------+
only showing top 10 rows


In [0]:
# merge operation performed (update and insert)
deltaTable = DeltaTable.forName(spark, "customers")

deltaTable.alias("old") \
    .merge(
        new_df.alias("new"),
        "old.customer_id = new.customer_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#  final dataset
final_df = spark.table("customers")

final_df.show(10)

+-----------+------+--------+---+----------+
|customer_id|  name|    city|age|membership|
+-----------+------+--------+---+----------+
|        107|Sanjay|  Kanpur| 31|      Gold|
|        154| Sonia|  Mumbai| 33|      Gold|
|        186|  Riya|Guwahati| 26|    Bronze|
|        122| Divya|  Mysuru| 25|      Gold|
|        124| Mohit|Guwahati| 24|    Bronze|
|        132| Arjun|    Pune| 24|      Gold|
|        134| Tanvi|   Surat| 35|      Gold|
|        137|  Neha|    Pune| 31|    Bronze|
|        142|Simran|  Nashik| 31|    Silver|
|        145| Arjun|   Noida| 33|    Bronze|
+-----------+------+--------+---+----------+
only showing top 10 rows


In [0]:
# total row count
print("Total Rows:", final_df.count())

Total Rows: 120


In [0]:
print("Initial Rows:", master_df.count())

Initial Rows: 101


In [0]:
# checking for nay duplicates in the dataset
final_df.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In this notebook used a customer_master csv dataset and stored it as a Delta table in Databricks.Then cleaned the data by removing null values and duplicate records. Next created another dataset customer_incremental to simulate new incoming customer data and used the Delta Lake (MERGE) operation to update existing records while inserting new ones. Finally verified the results by checking the total number of records and ensuring there were no duplicate customer ids. Understood how incremental data processing works using Delta Lake in Databricks.